# SQL-слой проекта LOL: DuckDB

В этом ноутбуке строим аналитический SQL-слой поверх общей таблицы `all_matches_common.csv`.

Зачем это нужно:

- показать владение SQL;
- не анализировать только через `read_csv`;
- подготовить агрегированные витрины для будущего дашборда;
- использовать связку, которую рекомендовал наставник: файлы данных + DuckDB.


## 0. Установка DuckDB

Если `duckdb` уже установлен в выбранном kernel, ячейка ничего существенного не изменит.


In [ ]:
%pip install -q duckdb


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\fiery\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Импорт библиотек и пути


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "normalized" / "all_matches_common.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "sql"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


DATA_PATH: c:\Users\fiery\OneDrive\Рабочий стол\учеба, фото\Дата аналитик\Проект LOL\data\normalized\all_matches_common.csv
OUTPUT_DIR: c:\Users\fiery\OneDrive\Рабочий стол\учеба, фото\Дата аналитик\Проект LOL\outputs\sql


## 2. Подключение DuckDB и регистрация CSV

DuckDB умеет читать CSV напрямую. Создаем view `matches_common`, чтобы дальше писать обычные SQL-запросы.


In [ ]:
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW matches_common AS
SELECT *
FROM read_csv_auto(
    '{DATA_PATH.as_posix()}',
    header = true,
    all_varchar = false
)
""")

con.execute("DESCRIBE matches_common").df()


,column_name,column_type,null,key,default,extra
0,data_source,VARCHAR,YES,None,None,None
1,match_id,VARCHAR,YES,None,None,None
2,source_tier,VARCHAR,YES,None,None,None
3,game_start_utc,TIMESTAMP,YES,None,None,None
4,game_duration_sec,BIGINT,YES,None,None,None
5,game_duration_min,DOUBLE,YES,None,None,None
6,game_mode,VARCHAR,YES,None,None,None
7,game_type,VARCHAR,YES,None,None,None
8,game_version,VARCHAR,YES,None,None,None
9,map_id,BIGINT,YES,None,None,None


## 3. Базовая проверка объема данных


In [ ]:
source_overview_sql = con.execute("""
SELECT
    data_source,
    COUNT(*) AS rows,
    COUNT(DISTINCT match_id) AS matches,
    COUNT(DISTINCT puuid) AS players,
    COUNT(DISTINCT champion_name) AS champions,
    AVG(game_duration_min) AS avg_game_duration_min
FROM matches_common
GROUP BY data_source
ORDER BY rows DESC
""").df()

source_overview_sql


,data_source,rows,matches,players,champions,avg_game_duration_min
0,kaggle,21910,2191,14294,169,28.250997
1,riot_api,3570,357,1707,171,26.461158


### Вывод

Этот запрос показывает, что мы можем получать базовые агрегаты через SQL. Kaggle дает основной объем данных, Riot API — свежую собственную выборку.


## 4. Полные матчи


In [ ]:
match_quality_sql = con.execute("""
WITH match_counts AS (
    SELECT
        data_source,
        match_id,
        COUNT(*) AS participants
    FROM matches_common
    GROUP BY data_source, match_id
)
SELECT
    data_source,
    participants,
    COUNT(*) AS matches
FROM match_counts
GROUP BY data_source, participants
ORDER BY data_source, participants
""").df()

match_quality_sql


,data_source,participants,matches
0,kaggle,10,2191
1,riot_api,10,357


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW complete_matches AS
WITH match_counts AS (
    SELECT
        data_source,
        match_id,
        COUNT(*) AS participants
    FROM matches_common
    GROUP BY data_source, match_id
)
SELECT m.*
FROM matches_common AS m
JOIN match_counts AS c
    ON m.data_source = c.data_source
    AND m.match_id = c.match_id
WHERE c.participants = 10
""")

con.execute("""
CREATE OR REPLACE VIEW complete_position_matches AS
SELECT *
FROM complete_matches
WHERE team_position IN ('TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY')
""")

con.execute("""
SELECT
    data_source,
    COUNT(*) AS rows,
    COUNT(DISTINCT match_id) AS full_matches
FROM complete_matches
GROUP BY data_source
ORDER BY rows DESC
""").df()


,data_source,rows,full_matches
0,kaggle,21910,2191
1,riot_api,3570,357


### Вывод

Для SQL-витрин используем `complete_matches`, чтобы исключить несколько неполных матчей Kaggle.

Для ролевых расчетов используем `complete_position_matches`: в ней оставлены только стандартные позиции `TOP`, `JUNGLE`, `MIDDLE`, `BOTTOM`, `UTILITY`. Это убирает `Invalid` из ARAM/нестандартных матчей Kaggle.


## 5. SQL-витрина по ролям


In [ ]:
role_stats_sql = con.execute("""
SELECT
    data_source,
    team_position,
    COUNT(*) AS rows,
    COUNT(DISTINCT match_id) AS matches,
    AVG(kda) AS avg_kda,
    AVG(damage_per_min) AS avg_damage_per_min,
    AVG(gold_per_min) AS avg_gold_per_min,
    AVG(vision_per_min) AS avg_vision_per_min
FROM complete_position_matches
GROUP BY data_source, team_position
ORDER BY data_source, team_position
""").df()

role_stats_sql


,data_source,team_position,rows,matches,avg_kda,avg_damage_per_min,avg_gold_per_min,avg_vision_per_min
0,kaggle,BOTTOM,4381,2191,3.275733,815.034292,424.155335,0.531789
1,kaggle,JUNGLE,4382,2191,3.846880,666.293238,409.161156,0.707717
2,kaggle,MIDDLE,4378,2191,3.358897,830.476817,405.115966,0.559496
3,kaggle,TOP,4378,2191,2.660946,799.082816,399.354207,0.528061
4,kaggle,UTILITY,4380,2191,4.032498,422.281440,294.019752,1.939351
5,riot_api,BOTTOM,714,357,3.547214,881.255553,499.620128,0.690643
6,riot_api,JUNGLE,714,357,4.520875,692.334006,462.723972,0.974162
7,riot_api,MIDDLE,714,357,3.566755,838.793257,437.049975,0.715870
8,riot_api,TOP,714,357,2.770413,763.428550,412.875679,0.817854
9,riot_api,UTILITY,714,357,4.226034,387.820238,313.930863,2.872299


## 6. SQL-витрина победителей и проигравших


In [ ]:
result_stats_sql = con.execute("""
SELECT
    data_source,
    win,
    CASE WHEN win THEN 'win' ELSE 'loss' END AS result,
    COUNT(*) AS rows,
    AVG(kda) AS avg_kda,
    AVG(damage_per_min) AS avg_damage_per_min,
    AVG(gold_per_min) AS avg_gold_per_min,
    AVG(vision_per_min) AS avg_vision_per_min,
    AVG(kills) AS avg_kills,
    AVG(deaths) AS avg_deaths,
    AVG(assists) AS avg_assists
FROM complete_matches
GROUP BY data_source, win
ORDER BY data_source, win
""").df()

result_stats_sql


,data_source,win,result,rows,avg_kda,avg_damage_per_min,avg_gold_per_min,avg_vision_per_min,avg_kills,avg_deaths,avg_assists
0,kaggle,False,loss,10955,1.666126,638.591104,354.517542,0.821321,4.697307,7.210132,6.412232
1,kaggle,True,win,10955,5.200878,773.995350,418.055816,0.884538,7.193154,4.715746,10.211228
2,riot_api,False,loss,1785,1.656317,644.882560,389.329868,1.171569,4.064426,6.579832,5.968627
3,riot_api,True,win,1785,5.796199,780.570081,461.150379,1.256762,6.567507,4.081793,10.021849


## 7. SQL-витрина по чемпионам


In [ ]:
champion_stats_sql = con.execute("""
SELECT
    data_source,
    champion_name,
    COUNT(DISTINCT match_id) AS games,
    SUM(CASE WHEN win THEN 1 ELSE 0 END) AS wins,
    AVG(CASE WHEN win THEN 1.0 ELSE 0.0 END) AS winrate,
    AVG(kda) AS avg_kda,
    AVG(damage_per_min) AS avg_damage_per_min,
    AVG(gold_per_min) AS avg_gold_per_min,
    AVG(vision_per_min) AS avg_vision_per_min
FROM complete_matches
GROUP BY data_source, champion_name
ORDER BY data_source, games DESC
""").df()

champion_stats_sql.head(20)


,data_source,champion_name,games,wins,winrate,avg_kda,avg_damage_per_min,avg_gold_per_min,avg_vision_per_min
0,kaggle,MissFortune,423,223.0,0.527187,3.181977,832.113510,430.825415,0.573770
1,kaggle,Caitlyn,421,207.0,0.491686,2.990522,773.478939,420.242095,0.531117
2,kaggle,Jinx,411,205.0,0.498783,3.096933,791.043065,421.495636,0.515148
3,kaggle,Lulu,405,202.0,0.498765,5.012141,286.913211,282.236897,2.032751
4,kaggle,Viego,362,186.0,0.513812,3.907700,645.158412,417.695375,0.700771
5,kaggle,Jhin,358,194.0,0.541899,4.182931,761.490263,429.660912,0.513592
6,kaggle,Lux,345,180.0,0.521739,3.953264,769.169887,358.340397,1.170081
7,kaggle,MonkeyKing,324,176.0,0.543210,4.602659,666.259918,415.245994,0.719007
8,kaggle,Viktor,312,154.0,0.493590,3.262779,924.254171,402.755659,0.544875
9,kaggle,Ezreal,299,154.0,0.515050,3.481071,893.546230,408.202708,0.498434


In [ ]:
champion_filtered_sql = con.execute("""
WITH champion_stats AS (
    SELECT
        data_source,
        champion_name,
        COUNT(DISTINCT match_id) AS games,
        SUM(CASE WHEN win THEN 1 ELSE 0 END) AS wins,
        AVG(CASE WHEN win THEN 1.0 ELSE 0.0 END) AS winrate,
        AVG(kda) AS avg_kda,
        AVG(damage_per_min) AS avg_damage_per_min,
        AVG(gold_per_min) AS avg_gold_per_min,
        AVG(vision_per_min) AS avg_vision_per_min
    FROM complete_matches
    GROUP BY data_source, champion_name
)
SELECT *
FROM champion_stats
WHERE
    (data_source = 'kaggle' AND games >= 30)
    OR
    (data_source = 'riot_api' AND games >= 10)
ORDER BY data_source, winrate DESC, games DESC
""").df()

champion_filtered_sql.head(30)


,data_source,champion_name,games,wins,winrate,avg_kda,avg_damage_per_min,avg_gold_per_min,avg_vision_per_min
0,kaggle,Taric,62,40.0,0.645161,5.698101,275.529708,300.981473,1.753124
1,kaggle,Shyvana,38,24.0,0.631579,3.460428,771.349048,429.881506,0.680622
2,kaggle,Briar,69,42.0,0.608696,3.140499,828.499218,432.112265,0.687174
3,kaggle,Fiora,55,33.0,0.600000,3.224787,863.325426,430.909961,0.510361
4,kaggle,Skarner,126,75.0,0.595238,5.562529,665.848944,374.034180,0.677602
5,kaggle,Milio,163,96.0,0.588957,5.850472,227.348626,274.818600,2.005279
6,kaggle,Udyr,67,39.0,0.582090,3.302161,605.695946,398.313431,0.721993
7,kaggle,Qiyana,64,37.0,0.578125,3.357174,787.584692,415.011961,0.613644
8,kaggle,Nilah,45,26.0,0.577778,2.844735,769.942835,439.875852,0.477179
9,kaggle,Maokai,156,90.0,0.576923,4.007129,605.552360,324.729214,1.196574


### Вывод

Для чемпионов используем разные пороги по источникам: у Kaggle больше данных, поэтому порог выше; у Riot API выборка меньше, поэтому порог мягче.


## 8. Экспорт SQL-витрин


In [ ]:
source_overview_sql.to_csv(OUTPUT_DIR / "sql_source_overview.csv", index=False)
match_quality_sql.to_csv(OUTPUT_DIR / "sql_match_quality.csv", index=False)
role_stats_sql.to_csv(OUTPUT_DIR / "sql_role_stats.csv", index=False)
result_stats_sql.to_csv(OUTPUT_DIR / "sql_result_stats.csv", index=False)
champion_stats_sql.to_csv(OUTPUT_DIR / "sql_champion_stats.csv", index=False)
champion_filtered_sql.to_csv(OUTPUT_DIR / "sql_champion_filtered.csv", index=False)

print("SQL-витрины сохранены в:", OUTPUT_DIR)


SQL-витрины сохранены в: c:\Users\fiery\OneDrive\Рабочий стол\учеба, фото\Дата аналитик\Проект LOL\outputs\sql


## 9. Звёздная схема (star schema)

До этого мы работали с одной широкой таблицей. Теперь соберём из неё классическую **звёздную схему** — стандарт для BI и аналитики:

- **факт** `fact_participant` — одна строка = один игрок в одном матче, только числа (меры) и ключи;
- **измерения** (справочники) — `dim_champion`, `dim_match`, `dim_player`, `dim_role`.

Так описание чемпиона / матча / игрока хранится один раз, а не дублируется в каждой строке. Дашборд потом «крутит» меры из факта в разрезе измерений.

In [ ]:
# Справочник чемпионов с классами (tags) — статический источник Data Dragon.
CHAMP_PATH = PROJECT_ROOT / "data" / "reference" / "champions.csv"

# champions_ref: сырой справочник + выделяем основной класс (первый тег из tags).
con.execute(f"""
CREATE OR REPLACE TABLE champions_ref AS
SELECT
    CAST(champion_id AS BIGINT) AS champion_id,
    champion_name,
    title,
    tags,
    split_part(tags, ',', 1) AS primary_class
FROM read_csv_auto('{CHAMP_PATH.as_posix()}', header = true)
""")

# ---- ИЗМЕРЕНИЯ (справочники) ----

# dim_champion: кто такой чемпион. Ключ = champion_id.
con.execute("""
CREATE OR REPLACE TABLE dim_champion AS
WITH from_facts AS (
    SELECT DISTINCT
        TRY_CAST(champion_id AS BIGINT) AS champion_id,
        champion_name
    FROM complete_matches
    WHERE champion_id IS NOT NULL
)
SELECT
    f.champion_id,
    f.champion_name,
    r.title,
    COALESCE(r.primary_class, 'Unknown') AS primary_class
FROM from_facts AS f
LEFT JOIN champions_ref AS r ON f.champion_id = r.champion_id
""")

# dim_match: что за матч. Ключ = data_source + match_id.
con.execute("""
CREATE OR REPLACE TABLE dim_match AS
SELECT
    data_source,
    match_id,
    ANY_VALUE(game_start_utc)    AS game_start_utc,
    ANY_VALUE(game_duration_min) AS game_duration_min,
    ANY_VALUE(game_version)      AS game_version,
    ANY_VALUE(queue_id)          AS queue_id,
    ANY_VALUE(source_tier)       AS source_tier
FROM complete_matches
GROUP BY data_source, match_id
""")

# dim_player: кто игрок. Ключ = data_source + puuid.
con.execute("""
CREATE OR REPLACE TABLE dim_player AS
SELECT
    data_source,
    puuid,
    ANY_VALUE(source_tier)   AS source_tier,
    ANY_VALUE(summoner_name) AS summoner_name,
    COUNT(*)                 AS appearances
FROM complete_matches
WHERE puuid IS NOT NULL
GROUP BY data_source, puuid
""")

# dim_role: расшифровка позиции на русский. Ключ = team_position.
con.execute("""
CREATE OR REPLACE TABLE dim_role AS
SELECT
    role_key,
    CASE role_key
        WHEN 'TOP'     THEN 'Топ'
        WHEN 'JUNGLE'  THEN 'Лес'
        WHEN 'MIDDLE'  THEN 'Мид'
        WHEN 'BOTTOM'  THEN 'Бот / керри'
        WHEN 'UTILITY' THEN 'Саппорт'
        ELSE 'Не определена'
    END AS role_name_ru
FROM (SELECT DISTINCT team_position AS role_key FROM complete_matches)
""")

print("Измерения построены:")
for dim in ["dim_champion", "dim_match", "dim_player", "dim_role"]:
    n = con.execute(f"SELECT COUNT(*) FROM {dim}").fetchone()[0]
    print(f"  {dim}: {n} строк")

Измерения построены:
  dim_champion: 172 строк
  dim_match: 2548 строк
  dim_player: 16001 строк
  dim_role: 6 строк


In [ ]:
# ---- ФАКТ ----
# fact_participant: 1 строка = 1 игрок в 1 матче.
# Внутри только КЛЮЧИ (по ним соединяемся со справочниками) и МЕРЫ (числа для агрегации).
con.execute("""
CREATE OR REPLACE TABLE fact_participant AS
SELECT
    -- ключи
    data_source,
    match_id,
    participant_id,
    TRY_CAST(champion_id AS BIGINT) AS champion_id,
    puuid,
    team_position AS role_key,
    team_id,
    win,
    -- меры
    kills, deaths, assists, kda,
    gold_earned, gold_per_min,
    total_damage_dealt_to_champions, damage_per_min,
    vision_score, vision_per_min
FROM complete_matches
""")

# Проверка целостности: в факте ровно 10 строк на матч,
# поэтому fact / 10 должно совпасть с числом матчей в dim_match.
fact_rows = con.execute("SELECT COUNT(*) FROM fact_participant").fetchone()[0]
dim_matches = con.execute("SELECT COUNT(*) FROM dim_match").fetchone()[0]
print(f"fact_participant: {fact_rows} строк")
print(f"fact / 10 = {fact_rows // 10}, dim_match = {dim_matches}  ->",
      "OK" if fact_rows // 10 == dim_matches else "РАСХОЖДЕНИЕ")

fact_participant: 25480 строк
fact / 10 = 2548, dim_match = 2548  -> OK


### Как это соединяется

Теперь вместо одной плоской таблицы — **факт + 4 справочника**:

- `fact_participant` — числа (kills, kda, gold_per_min, win) и ключи-ссылки;
- `dim_champion`, `dim_match`, `dim_player`, `dim_role` — описания.

Чтобы получить, например, «winrate по классам чемпионов», мы **соединяем** (`JOIN`) факт со справочником `dim_champion` по ключу `champion_id`. Класс чемпиона хранится в одном месте, а не повторяется в каждой строке.

In [ ]:
# Пример 1: winrate по классам чемпионов.
# JOIN факта со справочником чемпионов по ключу champion_id —
# класс (Tank/Mage/Assassin/...) живёт в dim_champion, а не в каждой строке факта.
champion_class_winrate = con.execute("""
SELECT
    d.primary_class                            AS champion_class,
    COUNT(DISTINCT f.match_id)                 AS games,
    AVG(CASE WHEN f.win THEN 1.0 ELSE 0.0 END) AS winrate,
    AVG(f.kda)                                 AS avg_kda,
    AVG(f.damage_per_min)                      AS avg_damage_per_min
FROM fact_participant AS f
JOIN dim_champion AS d ON f.champion_id = d.champion_id
WHERE f.data_source = 'kaggle'
GROUP BY d.primary_class
ORDER BY winrate DESC
""").df()

champion_class_winrate

,champion_class,games,winrate,avg_kda,avg_damage_per_min
0,Tank,1682,0.504196,3.657317,584.560485
1,Mage,2084,0.503381,3.384214,813.292157
2,Fighter,2173,0.502161,3.100436,738.037556
3,Support,1747,0.500803,4.567443,344.342561
4,Marksman,2173,0.497242,3.201655,813.145442
5,Assassin,1199,0.481965,3.373650,764.790853


In [ ]:
# Пример 2: средние метрики по ролям с человекочитаемым названием роли.
# JOIN факта со справочником ролей по ключу role_key (team_position).
role_named_stats = con.execute("""
SELECT
    r.role_name_ru        AS role,
    COUNT(*)              AS rows,
    AVG(f.gold_per_min)   AS avg_gold_per_min,
    AVG(f.damage_per_min) AS avg_damage_per_min,
    AVG(f.vision_per_min) AS avg_vision_per_min
FROM fact_participant AS f
JOIN dim_role AS r ON f.role_key = r.role_key
WHERE f.data_source = 'kaggle'
  AND r.role_name_ru <> 'Не определена'
GROUP BY r.role_name_ru
ORDER BY avg_gold_per_min DESC
""").df()

role_named_stats

,role,rows,avg_gold_per_min,avg_damage_per_min,avg_vision_per_min
0,Бот / керри,4381,424.155335,815.034292,0.531789
1,Лес,4382,409.161156,666.293238,0.707717
2,Мид,4378,405.115966,830.476817,0.559496
3,Топ,4378,399.354207,799.082816,0.528061
4,Саппорт,4380,294.019752,422.281440,1.939351


In [ ]:
# Экспорт звёздной схемы. Эти файлы можно грузить в BI как связанные таблицы:
# факт в центре, справочники соединяются с ним по ключам.
STAR_DIR = OUTPUT_DIR / "star"
STAR_DIR.mkdir(parents=True, exist_ok=True)

for table in ["fact_participant", "dim_champion", "dim_match", "dim_player", "dim_role"]:
    con.execute(f"SELECT * FROM {table}").df().to_csv(STAR_DIR / f"{table}.csv", index=False)

champion_class_winrate.to_csv(STAR_DIR / "example_champion_class_winrate.csv", index=False)
role_named_stats.to_csv(STAR_DIR / "example_role_named_stats.csv", index=False)

print("Звёздная схема сохранена в:", STAR_DIR)

Звёздная схема сохранена в: c:\Users\fiery\OneDrive\Рабочий стол\учеба, фото\Дата аналитик\Проект LOL\outputs\sql\star


## 10. Итог

На этом этапе мы добавили SQL-слой проекта:

- DuckDB читает общую таблицу `all_matches_common.csv`;
- созданы view `matches_common` и `complete_matches` (только полные матчи на 10 игроков);
- через SQL построены витрины по источникам, ролям, результатам и чемпионам;
- собрана **звёздная схема**: факт `fact_participant` + измерения `dim_champion`, `dim_match`, `dim_player`, `dim_role`;
- примеры `JOIN` показывают winrate по классам чемпионов и метрики по ролям;
- витрины сохранены в `outputs/sql`, звёздная схема — в `outputs/sql/star`.

Этот слой — основа для дашборда (BI грузит факт и измерения как связанные таблицы) и заметный плюс в описании проекта.